In [24]:
import pandas as pd
import numpy as np
from itertools import combinations
from mlxtend.preprocessing import TransactionEncoder

In [25]:
df = pd.read_excel("online_retail.xlsx", nrows = 10)

# Remove missing critical fields
df = df.dropna(subset=['Description', 'CustomerID', 'InvoiceDate'])

# Convert types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)

# Create synthetic transaction ID
df['InvoiceID'] = (
    df['CustomerID'].astype(str) + "_" +
    df['InvoiceDate'].astype(str)
)

df.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceID
0,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,17850_2010-12-01 08:26:00
1,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,17850_2010-12-01 08:26:00
2,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,17850_2010-12-01 08:26:00
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,17850_2010-12-01 08:26:00
4,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,17850_2010-12-01 08:26:00


In [26]:
invoice_items = df.groupby('InvoiceID')['Description'].apply(list)

print("Total Transactions:", len(invoice_items))
invoice_items.head()

Total Transactions: 3


InvoiceID
13047_2010-12-01 08:34:00                      [ASSORTED COLOUR BIRD ORNAMENT]
17850_2010-12-01 08:26:00    [WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET...
17850_2010-12-01 08:28:00    [HAND WARMER UNION JACK, HAND WARMER RED POLKA...
Name: Description, dtype: object

In [27]:
df['Hour'] = df['InvoiceDate'].dt.hour

def get_time_group(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

invoice_time = df.groupby('InvoiceID')['Hour'].first().apply(get_time_group)

morning_baskets = invoice_items[invoice_time == "Morning"].tolist()
afternoon_baskets = invoice_items[invoice_time == "Afternoon"].tolist()
evening_baskets = invoice_items[invoice_time == "Evening"].tolist()
night_baskets = invoice_items[invoice_time == "Night"].tolist()

print("Morning:", len(morning_baskets))
print("Afternoon:", len(afternoon_baskets))
print("Evening:", len(evening_baskets))
print("Night:", len(night_baskets))

Morning: 3
Afternoon: 0
Evening: 0
Night: 0


In [28]:
def encode_baskets(baskets):
    if len(baskets) == 0:
        return pd.DataFrame()
    te = TransactionEncoder()
    te_array = te.fit(baskets).transform(baskets)
    return pd.DataFrame(te_array, columns=te.columns_)

encoded_data = {
    "Morning": encode_baskets(morning_baskets),
    "Afternoon": encode_baskets(afternoon_baskets),
    "Evening": encode_baskets(evening_baskets),
    "Night": encode_baskets(night_baskets)
}

In [29]:
def compute_support(df_encoded):
    return df_encoded.mean().to_dict()

def generate_MIS(item_support, beta, LS):
    MIS = {}
    for item, supp in item_support.items():
        MIS[item] = max(beta * supp, LS)
    return MIS

In [30]:
def mmis(df_encoded, beta, LS):

    if df_encoded.empty:
        return pd.DataFrame()

    N = len(df_encoded)
    item_support = compute_support(df_encoded)
    MIS = generate_MIS(item_support, beta, LS)

    sorted_items = sorted(MIS.keys(), key=lambda x: MIS[x])

    # L1 generation
    L1 = []
    frequent_itemsets = []

    for item in sorted_items:
        if item_support[item] >= MIS[item]:
            L1.append((item,))
            frequent_itemsets.append({
                'itemset': (item,),
                'support': item_support[item]
            })

    current_L = L1
    k = 2

    while len(current_L) > 1:

        candidates = []

        # JOIN STEP
        for i in range(len(current_L)):
            for j in range(i+1, len(current_L)):

                l1 = current_L[i]
                l2 = current_L[j]

                if l1[:-1] == l2[:-1]:

                    candidate = tuple(sorted(set(l1) | set(l2),
                                             key=lambda x: MIS[x]))

                    if len(candidate) == k:
                        candidates.append(candidate)

        # PRUNE STEP
        next_L = []

        for cand in candidates:

            support = (
                df_encoded[list(cand)]
                .all(axis=1)
                .sum() / N
            )

            min_mis = MIS[cand[0]]

            if support >= min_mis:
                next_L.append(cand)
                frequent_itemsets.append({
                    'itemset': cand,
                    'support': support
                })

        current_L = next_L
        k += 1

    return pd.DataFrame(frequent_itemsets)

In [31]:
def generate_rules(frequent_df, min_conf=0.6, min_lift=1.0):

    if frequent_df.empty:
        return pd.DataFrame()

    support_dict = {
        frozenset(row['itemset']): row['support']
        for _, row in frequent_df.iterrows()
    }

    rules = []

    for itemset in support_dict:
        if len(itemset) > 1:

            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):

                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent

                    if antecedent in support_dict and consequent in support_dict:

                        supp_A = support_dict[antecedent]
                        supp_C = support_dict[consequent]
                        supp_AC = support_dict[itemset]

                        confidence = supp_AC / supp_A
                        lift = confidence / supp_C

                        if confidence >= min_conf and lift > min_lift:
                            rules.append({
                                'antecedents': tuple(antecedent),
                                'consequents': tuple(consequent),
                                'support': supp_AC,
                                'confidence': confidence,
                                'lift': lift
                            })

    return pd.DataFrame(rules).sort_values(by='lift', ascending=False)

In [32]:
experiments = [
    {"name": "Exp 1", "beta": 0.5, "LS": 0.01},
    {"name": "Exp 2", "beta": 0.7, "LS": 0.02}
]

results = {}

for exp in experiments:

    print("\n" + "="*50)
    print(f"{exp['name']} | Beta={exp['beta']} | LS={exp['LS']}")
    print("="*50)

    for time_group, df_encoded in encoded_data.items():

        if df_encoded.empty:
            continue

        print(f"\n--- {time_group} ---")

        freq_items = mmis(df_encoded, exp['beta'], exp['LS'])

        print("Number of frequent itemsets:", len(freq_items))

        if not freq_items.empty:
            print("Largest itemset length:",
                  freq_items['itemset'].apply(len).max())

            print("\nTop 5 frequent itemsets:")
            print(freq_items.sort_values(
                'support', ascending=False).head(5))

        rules = generate_rules(freq_items)

        print("\nNumber of rules:", len(rules))

        if not rules.empty:
            print("\nTop 5 strongest rules:")
            print(rules.head(5))

        results[(exp['name'], time_group)] = {
            "freq": freq_items,
            "rules": rules
        }


Exp 1 | Beta=0.5 | LS=0.01

--- Morning ---
Number of frequent itemsets: 131
Largest itemset length: 7

Top 5 frequent itemsets:
                                itemset   support
0      (ASSORTED COLOUR BIRD ORNAMENT,)  0.333333
1     (CREAM CUPID HEARTS COAT HANGER,)  0.333333
2  (GLASS STAR FROSTED T-LIGHT HOLDER,)  0.333333
3          (HAND WARMER RED POLKA DOT,)  0.333333
4             (HAND WARMER UNION JACK,)  0.333333

Number of rules: 1934

Top 5 strongest rules:
                              antecedents  \
0    (GLASS STAR FROSTED T-LIGHT HOLDER,)   
1       (CREAM CUPID HEARTS COAT HANGER,)   
2  (KNITTED UNION FLAG HOT WATER BOTTLE,)   
3       (CREAM CUPID HEARTS COAT HANGER,)   
4       (RED WOOLLY HOTTIE WHITE HEART.,)   

                              consequents   support  confidence  lift  
0       (CREAM CUPID HEARTS COAT HANGER,)  0.333333         1.0   3.0  
1    (GLASS STAR FROSTED T-LIGHT HOLDER,)  0.333333         1.0   3.0  
2       (CREAM CUPID HEARTS COAT HAN